# Medibuddy Insurance – Analysis & Charges Prediction
Follows the Workflow.xlsx steps: import libraries → load both files → combine → gender vs charges histogram → mean charges. Then answers the 8 business questions and builds a tuned ML model.

`pip install pandas numpy matplotlib seaborn scipy scikit-learn openpyxl joblib`

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns, warnings, joblib
from scipy import stats
warnings.filterwarnings('ignore'); sns.set_theme(style='whitegrid')
PATH = ''   # folder holding the three files

## 1. Load & combine
Workflow.xlsx mentions `read_csv`/`concat`, but both source files are Excel and share `Policy no.`, so use `read_excel` + `merge` on the key (a plain concat would stack rows, not join columns).

In [ ]:
df1 = pd.read_excel(PATH+'Medibuddy_insurance_data_personal_details__1___2_.xlsx')   # Policy no., children, smoker, region
df2 = pd.read_excel(PATH+'Medibuddy_Insurance_Data_Price__1___2_.xlsx')              # Policy no., age, sex, bmi, charges in INR
df = df1.merge(df2, on='Policy no.', how='inner', validate='one_to_one')
df.columns = ['policy','children','smoker','region','age','sex','bmi','charges']
print(df.shape, '| missing:', df.isna().sum().sum(), '| duplicate policies:', df.policy.duplicated().sum())
df.head()

In [ ]:
df.describe(include='all').T

In [ ]:
df['bmi_cat'] = pd.cut(df.bmi,[0,18.5,25,30,100],labels=['Underweight','Normal','Overweight','Obese'],right=False)
df['age_band'] = pd.cut(df.age,[17,25,35,45,55,65],labels=['18-25','26-35','36-45','46-55','56-64'])
# Workflow step: gender & charges histogram
sns.histplot(data=df,x='charges',hue='sex',bins=40,kde=True,multiple='layer'); plt.title('Charges (INR) by gender'); plt.show()

## Q1. Does gender matter?

In [ ]:
print(df.groupby('sex').charges.agg(['count','mean','median']).round(0))
m,f = df[df.sex=='male'].charges, df[df.sex=='female'].charges
print('Welch t-test p =',round(stats.ttest_ind(m,f,equal_var=False).pvalue,4),'| Mann-Whitney p =',round(stats.mannwhitneyu(m,f).pvalue,4))
print(pd.crosstab(df.sex,df.smoker,normalize='index').round(3))
print(df.groupby(['sex','smoker']).charges.mean().unstack().round(0))

## Q2. Average spend per policy

In [ ]:
print('Mean :',round(df.charges.mean(),2),'| Median:',round(df.charges.median(),2))
print(df.groupby('smoker').charges.mean().round(0))
sns.boxplot(data=df,x='smoker',y='charges'); plt.show()

## Q3. Region

In [ ]:
print(df.groupby('region').charges.agg(['count','mean','median']).round(0))
print('ANOVA p =',round(stats.f_oneway(*[g.charges for _,g in df.groupby('region')]).pvalue,4))
print(df.groupby(['region','smoker']).charges.mean().unstack().round(0))
print(pd.crosstab(df.region,df.smoker,normalize='index').round(3)); print(df.groupby('region').bmi.mean().round(1))
sns.barplot(data=df,x='region',y='charges',hue='smoker'); plt.show()

## Q4. Dependents

In [ ]:
print(df.groupby('children').charges.agg(['count','mean','median']).round(0))
print('ANOVA p =',round(stats.f_oneway(*[g.charges for _,g in df.groupby('children')]).pvalue,4),'| corr =',round(df.children.corr(df.charges),3))
sns.pointplot(data=df,x='children',y='charges',hue='smoker'); plt.show()

## Q5 & Q8. BMI and discount simulation

In [ ]:
print(df.groupby('bmi_cat',observed=True).charges.agg(['count','mean','median']).round(0))
print(df.groupby(['bmi_cat','smoker'],observed=True).charges.mean().unstack().round(0))
print('BMI-charges corr  smokers:',round(df[df.smoker=='yes'][['bmi','charges']].corr().iloc[0,1],2),
      '| non-smokers:',round(df[df.smoker=='no'][['bmi','charges']].corr().iloc[0,1],2))
sns.scatterplot(data=df,x='bmi',y='charges',hue='smoker',alpha=.6); plt.axvline(30,ls='--',c='k'); plt.show()
# Q8: wellness-discount simulation (non-smokers only) – discount % applied to normal-BMI non-smokers
base = df.charges.sum(); normal_ns = df[(df.smoker=='no')&(df.bmi_cat=='Normal')]
for d in (0.05,0.10): print(f'{d:.0%} discount on Normal-BMI non-smokers costs INR {normal_ns.charges.sum()*d:,.0f} = {normal_ns.charges.sum()*d/base:.2%} of total spend')

## Q6. Smoker vs non-smoker

In [ ]:
print(df.groupby('smoker').charges.agg(['count','mean','median','sum']).round(0))
print('Welch t-test p =',stats.ttest_ind(df[df.smoker=='yes'].charges,df[df.smoker=='no'].charges,equal_var=False).pvalue)
print((df.groupby('smoker').charges.sum()/df.charges.sum()).round(3))

## Q7. Age

In [ ]:
print(df.groupby('age_band',observed=True).charges.agg(['count','mean','median']).round(0))
print('Pearson r =',round(stats.pearsonr(df.age,df.charges)[0],3))
print(df.groupby(['age_band','smoker'],observed=True).charges.mean().unstack().round(0))
sns.lmplot(data=df,x='age',y='charges',hue='smoker',scatter_kws={'s':8}); plt.show()

## Correlation heat-map

In [ ]:
num = df[['age','bmi','children','charges']].assign(smoker=(df.smoker=='yes').astype(int))
sns.heatmap(num.corr(),annot=True,cmap='coolwarm',fmt='.2f'); plt.show()

## 2. Machine-learning model
Feature engineering is driven by the EDA: a smoker × BMI interaction, an obesity flag (BMI ≥ 30) and an age² term. Test set = 20 % (stratified on smoker); all tuning uses 5-fold CV on the training set only so the test set stays untouched.

In [ ]:
from sklearn.model_selection import train_test_split, KFold, cross_val_score, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

def make_features(d):
    X = pd.DataFrame({'age':d.age,'bmi':d.bmi,'children':d.children,
                      'smoker':(d.smoker=='yes').astype(int),'male':(d.sex=='male').astype(int)})
    for r in ['northwest','southeast','southwest']: X['r_'+r] = (d.region==r).astype(int)
    X['obese'] = (d.bmi>=30).astype(int)
    X['smoker_obese'] = X.smoker*X.obese
    X['smoker_bmi']   = X.smoker*X.bmi
    X['age2'] = d.age**2
    return X
X, y = make_features(df), df.charges
Xtr,Xte,ytr,yte = train_test_split(X,y,test_size=.2,random_state=42,stratify=X.smoker)
kf = KFold(5,shuffle=True,random_state=42)

In [ ]:
def evaluate(name, model, results):
    cv = cross_val_score(model,Xtr,ytr,cv=kf,scoring='r2').mean()
    model.fit(Xtr,ytr); p = model.predict(Xte)
    results.append(dict(model=name,cv_r2=cv,test_r2=r2_score(yte,p),MAE=mean_absolute_error(yte,p),RMSE=mean_squared_error(yte,p)**.5))
results=[]
evaluate('Linear Regression',Pipeline([('s',StandardScaler()),('m',LinearRegression())]),results)
evaluate('Poly(2) Ridge',Pipeline([('p',PolynomialFeatures(2)),('s',StandardScaler()),('m',Ridge(10))]),results)
evaluate('Random Forest (default)',RandomForestRegressor(100,random_state=42),results)
evaluate('Gradient Boosting (default)',GradientBoostingRegressor(random_state=42),results)
pd.DataFrame(results).round(4)

### Hyper-parameter tuning (RandomizedSearchCV, 5-fold)
For a finer search raise `n_iter` or switch to `GridSearchCV` (slower).

In [ ]:
gb = RandomizedSearchCV(GradientBoostingRegressor(random_state=42),
    {'n_estimators':[100,200,300,500],'learning_rate':[.01,.02,.05,.1],'max_depth':[2,3,4],
     'min_samples_leaf':[10,20,40],'subsample':[.7,.9,1],'max_features':[.6,.8,1.0]},
    n_iter=30,cv=kf,scoring='r2',random_state=42,n_jobs=-1).fit(Xtr,ytr)
rf = RandomizedSearchCV(RandomForestRegressor(random_state=42),
    {'n_estimators':[100,200,400],'max_depth':[4,5,6,8],'min_samples_leaf':[5,10,20],'max_features':[.6,.8,1.0]},
    n_iter=20,cv=kf,scoring='r2',random_state=42,n_jobs=-1).fit(Xtr,ytr)
print('GB best:',gb.best_params_,round(gb.best_score_,4)); print('RF best:',rf.best_params_,round(rf.best_score_,4))
for n,m in [('GB tuned',gb),('RF tuned',rf)]:
    p=m.predict(Xte); results.append(dict(model=n,cv_r2=m.best_score_,test_r2=r2_score(yte,p),MAE=mean_absolute_error(yte,p),RMSE=mean_squared_error(yte,p)**.5))
res = pd.DataFrame(results).round(4); res

In [ ]:
best = gb.best_estimator_; p = best.predict(Xte)
fig,ax = plt.subplots(1,3,figsize=(17,4.5))
ax[0].scatter(yte,p,s=10,alpha=.6); ax[0].plot([0,yte.max()],[0,yte.max()],'r--'); ax[0].set(title='Actual vs predicted',xlabel='Actual',ylabel='Predicted')
ax[1].scatter(p,yte-p,s=10,alpha=.6); ax[1].axhline(0,c='r'); ax[1].set(title='Residuals',xlabel='Predicted')
pd.Series(best.feature_importances_,index=X.columns).sort_values().plot.barh(ax=ax[2],title='Feature importance'); plt.tight_layout(); plt.show()

In [ ]:
# Learning curve: is more data / more complexity likely to help?
from sklearn.model_selection import learning_curve
sz,tr,va = learning_curve(best,Xtr,ytr,cv=kf,scoring='r2',train_sizes=np.linspace(.2,1,6))
plt.plot(sz,tr.mean(1),'o-',label='train'); plt.plot(sz,va.mean(1),'o-',label='CV'); plt.legend(); plt.title('Learning curve'); plt.show()
joblib.dump(best,'medibuddy_gb_model.pkl')
# example: predict a new policy
new = pd.DataFrame([dict(age=45,sex='male',bmi=32.0,children=2,smoker='no',region='southeast')])
print('Predicted charges (INR):', round(best.predict(make_features(new))[0]))